# Campeonato de Algoritmos - MLOps Hyperparameter Tuning

In [ ]:
import pandas as pd
import boto3
import io
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

import warnings
warnings.filterwarnings('ignore')

# Convenções Corporativas Seguras MLOps AWS
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"

mlflow.set_tracking_uri("http://mlflow-server:5000")
mlflow.set_experiment("Penguins_Classification")
print("MLflow Tracking URI Ativado!")


In [ ]:
# 1. Extração Segura da Landing Zone (Raw Storage)
s3_client = boto3.client(
    's3', endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    region_name='us-east-1'
)

response = s3_client.get_object(Bucket='landing-zone', Key='penguins.csv')
df = pd.read_csv(io.BytesIO(response['Body'].read()))
df.dropna(inplace=True)
print("Dados Injetados no Jupyter!")
df.head(2)

In [ ]:
# 2. Computação Dummies e Particionamento
X = pd.get_dummies(df.drop('especies', axis=1))
y = df['especies']

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)
print("Formato dos dados Treino/Teste:")
print(X_treino.shape)

In [ ]:
# 3. Campeonato de Algoritmos (Nested Runs no MLflow)

# Seleção do Cientista (Space Search)
modelos_para_testar = [
    ("DecisionTree_Dep3", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("DecisionTree_Dep10", DecisionTreeClassifier(max_depth=10, random_state=42)),
    ("RandomForest_10", RandomForestClassifier(n_estimators=10, random_state=42)),
    ("RandomForest_100", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("LogisticRegression", LogisticRegression(max_iter=500, random_state=42)),
    ("SVC_Kernel_Linear", SVC(kernel="linear", random_state=42))
]

with mlflow.start_run(run_name="Torneio_Pinguins_V1") as parent_run:
    print(f"🏆 Iniciando Torneio Pai (RunID: {parent_run.info.run_id})\n")
    
    for nome_modelo, modelo in modelos_para_testar:
        # A flag nested=True agrupa os modelos visualmente na tela do MLflow
        with mlflow.start_run(run_name=nome_modelo, nested=True) as child_run:
            print(f"⏳ Treinando e Avaliando -> {nome_modelo}...")
            
            # --- Ajuste do Algoritmo ---
            modelo.fit(X_treino, y_treino)
            
            # --- Avaliação ---
            previsoes = modelo.predict(X_teste)
            acuracia = accuracy_score(y_teste, previsoes)
            f1 = f1_score(y_teste, previsoes, average='weighted')
            
            # --- Registros (MLflow) ---
            mlflow.log_param("algoritmo", type(modelo).__name__)
            # log_params faz varredura automática de "todos os parametros de hyper-tunings":
            mlflow.log_params(modelo.get_params()) 
            
            mlflow.log_metric("acuracia", acuracia)
            mlflow.log_metric("f1_score", f1)
            
            # --- Assinatura e Injeção S3 ---
            from mlflow.models.signature import infer_signature
            signature = infer_signature(X_treino, modelo.predict(X_treino))
            
            mlflow.sklearn.log_model(
                sk_model=modelo,
                artifact_path="modelo_penguins",
                signature=signature
            )
            print(f"  > F1-Score: {f1:.4f} | Rastreamento salvo!")

print("\n✅ Torneio Finalizado! Vá ao MLflow (http://localhost:5000)")
print("👉 Ordene por 'f1_score', escolha a versão perfeita e clique em 'Register Model'.")